Question 1

Now for the finale: You need to code your own backtesting agent/engine. The engine will take the dataset as an input. Note that you will take your initial capital as 1,00,000 INR and will use all your capital in a long trade.

Derivables: Your backtesting agent shall perform the operations of long trades, stop loss and take profit. Clearly mention the percentage of stop loss/take profit that you fix in your backtesting engine as a comment.

Your backtesting framework should print the net profit, sharpe ratio, maximum draw- down, overall total trades, total number of winning/losing trades (Do check that the number of winning and losing trades combined should be the total number of trades).

Hints: You will need a lot of memory (multiple arrays) for the backtesting engine. Few of them will be your everyday portfolio value, return of each trade, boolean variable to keep track of an ongoing trade.

In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = yf.download('TCS.NS', start='2023-01-01', end='2023-12-31')
df.columns = df.columns.get_level_values(0)


/tmp/ipython-input-359566667.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('TCS.NS', start='2023-01-01', end='2023-12-31')
[*********************100%***********************]  1 of 1 completed


In [3]:
def bollinger_bands(df):
  df['sma']=df['Close'].rolling(20).mean()
  df['sd']=df['Close'].rolling(20).std()
  df['upper_band'] = df['sma'] + 2*df['sd']
  df['lower_band'] = df['sma'] - 2*df['sd']
  return df

def rsi(df, period=14):
  df['diff'] = df['Close'].diff()
  df['Gain'] = (np.where(df['diff'] < 0, 0, df['diff']))
  df['Loss'] = abs(np.where(df['diff'] > 0, 0 , df['diff']))
  df['avg_gain'] = df['Gain'].rolling(period).mean()
  df['avg_loss'] = df['Loss'].rolling(period).mean()
  df['RS'] = df['avg_gain']/df['avg_loss']
  df['RSI'] = 100 - (100/(1+df['RS']))
  return df

def macd(df, short_period=12, long_period=26):
  df['short_ema'] = df['Close'].ewm(short_period).mean()
  df['long_ema'] = df['Close'].ewm(long_period).mean()
  df['macd'] = df['short_ema']-df['long_ema']
  df['signal'] = df['macd'].ewm(9).mean()
  df['macd_signal'] = df['macd']-df['signal']
  return df

def so(df, k_period=12, d_period=9):
  df['lowest_low'] = df['Low'].rolling(k_period).min()
  df['highest_high'] = df['High'].rolling(k_period).max()

  df['%K'] = 100*(df['Close']-df['lowest_low'])/(df['highest_high']-df['lowest_low'])
  df['%D'] = df['%K'].rolling(d_period).mean()

  return df

def atr(df, period=14):
  df['h-l'] = df['High'] - df['Low']
  df['h-prev_close'] = abs(df['High'] - df['Close'].shift(1))
  df['l-prev_close'] = abs(df['Low'] - df['Close'].shift(1))
  df['true_range'] = df[['h-l', 'h-prev_close', 'l-prev_close']].max(axis=1)

  df['ATR'] = df['true_range'].ewm(period).mean()
  return df








In [4]:
def signal_bollinger(df):
  df = bollinger_bands(df)
  # 1:buy, -1:sell
  df['Signal'] = 0
  df.loc[df['Close'] <= df['lower_band'], 'Signal'] = 1
  df.loc[df['Close'] >= df['upper_band'], 'Signal'] = -1
  return df


def signal_rsi(df, oversold_threshold=30, overbought_threshold=70):
  df = rsi(df)
  # 1:buy, -1:sell
  df['Signal'] = 0
  df.loc[df['RSI'] <= oversold_threshold, 'Signal'] = 1
  df.loc[df['RSI'] >= overbought_threshold, 'Signal'] = -1
  return df


def signal_macd(df):
  df = macd(df)
  # 1:buy, -1:sell
  df['Signal'] = 0
  df.loc[df['macd_signal'] > 0, 'Signal'] = 1
  df.loc[df['macd_signal'] < 0, 'Signal'] = -1
  return df


def signal_so(df):
  df = so(df)
  # 1:buy, -1:sell
  df['Signal'] = 0
  df.loc[(df['%K'] > df['%D']) & (df['%K'] > 80), 'Signal'] = 1
  df.loc[(df['%K'] < df['%D']) & (df['%K'] < 20), 'Signal'] = -1
  return df







In [37]:
def backtesting(df, initial_capital, signal):
  capital = initial_capital
  position = 0 #checks wether i am in the market
  entry_price = 0
  pnl = [] #profit or loss of a trade
  daily_rec = [] #stores day by day portfolio value
  daily_return = [] #stores day by day return

  #trade history
  total_trade = 0
  total_win = 0
  total_loss = 0

  stop_loss = -0.02
  take_profit = 0.04

  for day in range(len(df)):
    price = float(df.iloc[day]["Close"])

    if position == 0:

      #check buy condition
      if signal[day] == 1:
        #buy share
        shares = capital//price
        if shares > 0:
          #set entry price
          entry_price = price
          #set position (no. of shares)
          position = shares
          #reduce capital
          capital -= shares * price
          #increase total trade
          total_trade += 1

    else:
      #calculate current return
      current_return = (price - entry_price)/entry_price
      #check if return < stop loss or reurn > take profit
      if current_return < stop_loss or current_return > take_profit:
        #sell all shares
        capital += position * price
        #store trade loss/profit, increase total_win/total_loss
        p_l = position * (price - entry_price)
        pnl.append(p_l)
        if price > entry_price:
          total_win += 1
        elif price < entry_price:
          total_loss += 1
        #set position to 0
        position = 0

    if position > 0:
      portfolio_value = capital + shares * price
    else:
      portfolio_value = capital

    daily_rec.append(portfolio_value)

    if day == len(df)-1 and position > 0:
      #sell all shares, add money to capital
      capital += position*price
      #store trade loss/profit, increase total_win
      p_l = position * (price - entry_price)
      pnl.append(p_l)
      #set position to 0
      position = 0

  net_profit = capital - initial_capital

  # calculate sharpe ratio
  daily_return = []
  for i in range(1,len(daily_rec)-1):
    day_return = (daily_rec[i] - daily_rec[i-1])/daily_rec[i-1]
    daily_return.append(day_return)

  daily_ret = np.array(daily_return)
  sharpe = np.mean(daily_ret)/np.std(daily_ret)

  daily_rec = np.array(daily_rec)
  peak = np.maximum.accumulate(daily_rec)
  drawdown = (daily_rec - peak)/peak
  max_drawdown = np.min(drawdown)

  return net_profit, sharpe, max_drawdown, total_trade, total_win, total_loss



In [38]:
#generate signal
df = signal_rsi(df)
#convert signal column to list, to use in backtest function
signal = df["Signal"].tolist()

net_profit, sharpe, max_drawdown, total_trades, total_win, total_loss = backtesting(df, 100000, signal)

print("net profit:",net_profit, "\nsharpe ratio:", sharpe, "\nmaximum drawdown:", max_drawdown, "\noverall total trades", total_trades, "\ntotal number of winning/losing trades:", total_win, "/", total_loss)


net profit: 9540.56494140625 
sharpe ratio: 0.07819140356385187 
maximum drawdown: -0.07884702336441676 
overall total trades 5 
total number of winning/losing trades: 3 / 2
